# 北尾早霧・砂川武貴・山田知明『定量的マクロ経済学と数値計算』日本評論社
## 第3章：動的計画法
* DPをパラメトリックに解く

In [26]:
# import Pkg; Pkg.add("PlotlyBase")

In [27]:
using Dierckx
using Optim
using Plots
plotly()
# pyplot()

Plots.PlotlyBackend()

In [28]:
using BenchmarkTools

In [29]:
# グリッドを生成するためのモジュールを読み込む
include("../quant_macro_book/chapter3/Julia/Jupyter/GenerateGrid.jl")

Main.GenerateGrid

### カリブレーション
* パラメータをひとつの変数にまとめる：structを使う

In [30]:
struct Params
    # パラメータ
    β::Float64 #割引因子
    γ::Float64 #相対的危険回避度
    α::Float64 #資本分配率
    δ::Float64 #固定資本減耗

    # グリッド
    nk::Int64 #資本グリッドの数
    kmax::Float64 #資本グリッドの最大値
    kmin::Float64 #資本グリッドの最小値
    kgrid::Vector{Float64} #資本グリッド

    # 繰り返し計算
    maxit::Int64 # 繰り返し計算の最大値
    tol::Float64 # 計算誤差の許容値(tolerance of error)
end

In [31]:
function calibration()
    β = 0.96
    γ = 1.0
    α = 0.4
    δ = 1.0 # 0.08

    nk = 101
    kmax = 0.5 # 固定資本減耗が0.08の場合、ここを10.0にする
    kmin = 0.05

    # 自作のコードで等分のグリッドを計算
    kgrid = GenerateGrid.grid_uni(kmin, kmax, nk)
    # これまで通り⬇でもOK
    #kgrid = collect(LinRange(kmin, kmax, nk))

    maxit = 1000
    tol = 1e-5 # 2.収束の基準を設定

    return Params(β, γ, α, δ, nk, kmax, kmin, kgrid, maxit, tol)
end

calibration (generic function with 1 method)

In [32]:
params = calibration();

---

## 経済学でよく使う関数

In [33]:
include("../quant_macro_book/chapter3/Julia/Jupyter/MyEconFcn.jl")

Main.MyEconFcn

---

## VFIのための準備：当て推量
* 価値関数の初期値はすべての資源を使い切った場合の効用：有限期間モデルの最終期をイメージ
    * この形である必要はまったくない

In [34]:
# 価値関数と政策関数の初期値
pfcn0 = zeros(params.nk) # すべての資源を使い切る
vfcn0 = MyEconFcn.crra.(params.kgrid.^params.α + (1-params.δ)*params.kgrid, params.γ) # すべてを消費する際の効用

pfcn1 = zeros(params.nk)
vfcn1 = zeros(params.nk)

# 繰り返し誤差を保存する変数を設定
# 途中経過を図示する目的なので、通常は不要(むしろ遅くなるので消すべき)
dif = zeros(2, params.maxit);

In [35]:
# 利用可能な資産をあらかじめ計算しておく
wealth = params.kgrid.^params.α + (1-params.δ)*params.kgrid;

### Optimを使うための関数(ベルマン方程式)を設定

In [36]:
"""
k'を1つ与えた際にベルマン方程式の値を返す

### Inputs
`params::Params`: パラメータなどを含むオブジェクト
`wealth::Float64`: 今期利用可能な資産
`kprime::Float64`: 次期の資本量
`vnext::Spline1D`: 次期の価値関数をスプライン近似した際の係数

### Outputs 
`value::Float64`:　負値にしたベルマン方程式
"""
function BellmanEq(params::Params, wealth::Float64, kprime::Float64, vnext::Spline1D)
    value = MyEconFcn.crra((wealth - kprime), params.γ) + params.β*vnext(kprime)
    value = -1*value
    return value 
end

BellmanEq

In [37]:
# 価値関数を繰り返し計算
for it = 1:params.maxit

    # 次期の価値関数を補間
    #vnext = Spline1D(params.kgrid, vfcn0, k=1, bc="extrapolate") #線形補間
    vnext = Spline1D(params.kgrid, vfcn0, k=3, bc="extrapolate") #スプライン補間

    for i = 1:params.nk
        BellmanEq!(kprime) = BellmanEq(params, wealth[i], kprime, vnext)
        res = optimize(BellmanEq!, 0.0, wealth[i], GoldenSection()) # 最適化
        pfcn1[i] = res.minimizer
        vfcn1[i] = -res.minimum # 最小値を探していたので符号を反転させる
    end

    dif1 = maximum(abs.((vfcn1 - vfcn0)./vfcn0)) # 価値関数の繰り返し計算誤差
    dif2 = maximum(abs.((pfcn1 - pfcn0)./pfcn0)) # 政策関数の繰り返し計算誤差(図示のため)

    # 収束途中の繰り返し計算誤差を保存
    dif[1, it] = dif1
    dif[2, it] = dif2

    vfcn0 = deepcopy(vfcn1)
    pfcn0 = deepcopy(pfcn1)

    # println("iteration counter: $it")
    # println("error (value): $dif1")
    # println("error (policy): $dif2")
    # println()

    if dif1 < params.tol
        break
    end

    if it == params.maxit
        println("The model does not converge...")
    end
end

---

## 計算速度を測るためにNumerical DPを関数化

In [38]:
"""
状態変数のみ離散化して操作変数は連続的に値を取る場合の動的計画法(parametric DP)の解法.
アルゴリズムの詳細は、Johnson et al. (1993)を参照

### Inputs
`params::Params`: パラメータ等を含む構造体

### Outputs
`vfcn0::Vector{Float64}`: 計算によって得られた価値関数
`pfcn1::Vector{Float64}`: 計算によって得られた政策関数
"""
function pdp(params::Params, conv_criterion::String; verbosity=1)

    # 価値関数と政策関数の初期化
    pfcn0 = zeros(params.nk)
    vfcn0 = MyEconFcn.crra.(params.kgrid.^params.α + (1-params.δ)*params.kgrid, params.γ)

    pfcn1 = zeros(params.nk)
    vfcn1 = zeros(params.nk)

    # 利用可能な資産をあらかじめ計算しておく
    wealth = params.kgrid.^params.α + (1-params.δ)*params.kgrid

    # 繰り返し誤差を保存する変数を設定
    # 途中経過を図示する目的なので、通常は不要(むしろ遅くなるので消すべき)
    dif = zeros(2, params.maxit)

    # 価値関数を繰り返し計算
    for it = 1:params.maxit

        #次期の価値関数を補間
        #vnext = Spline1D(params.kgrid, vfcn0, k=1, bc="extrapolate") #線形補間
        vnext = Spline1D(params.kgrid, vfcn0, k=3, bc="extrapolate") #スプライン補間

        for i = 1:params.nk
            BellmanEq!(kprime) = BellmanEq(params, wealth[i], kprime, vnext)
            res = optimize(BellmanEq!, 0.0, wealth[i], GoldenSection()) # 最適化
            pfcn1[i] = res.minimizer
            vfcn1[i] = -res.minimum # 最小値を探していたので符号を反転させる
        end

        dif1 = maximum(abs.((vfcn1 - vfcn0)./vfcn0)) # 価値関数の繰り返し計算誤差
        dif2 = maximum(abs.((pfcn1 - pfcn0)./pfcn0)) # 政策関数の繰り返し計算誤差(図示のため)

        # 収束途中の繰り返し計算誤差を保存
        dif[1, it] = dif1
        dif[2, it] = dif2

        vfcn0 = deepcopy(vfcn1)
        pfcn0 = deepcopy(pfcn1)

#         println("iteration counter: $it")
#         println("error (value): $dif1")
#         println("error (policy): $dif2")
#         println()

        if conv_criterion == "vf"
            if dif1 < params.tol
                if verbosity == 1
                    println("iteration counter: $it")
                    println("error (value): $dif1")
                    println("error (policy): $dif2")
                end
                break
            end
        elseif conv_criterion == "pf"
            if dif2 < params.tol
                if verbosity == 1
                    println("iteration counter: $it")
                    println("error (value): $dif1")
                    println("error (policy): $dif2")
                end
                break
            end
        end

        if it == params.maxit
            println("The model does not converge...")
        end
    end

    return vfcn0, pfcn0, dif
end

pdp

In [14]:
@benchmark pdp(params, "vf"; verbosity=0)

BenchmarkTools.Trial: 158 samples with 1 evaluation per sample.
 Range (min … max):  30.804 ms …  37.441 ms  ┊ GC (min … max): 0.50% … 0.72%
 Time  (median):     31.429 ms               ┊ GC (median):    0.49%
 Time  (mean ± σ):   31.738 ms ± 872.573 μs  ┊ GC (mean ± σ):  0.59% ± 0.63%

       ▂▄▆█▄▅▆▅▂▃                                               
  ▅▁▇▅▁██████████▅▇▁▅▁▅▅▁▅▅▁▁█▁▅▇▁█▅▇▅▅▅▅▅▅█▅▇▁▇▁▅▁▁▁▁▁▁▁▅▁▁▁▅ ▅
  30.8 ms       Histogram: log(frequency) by time      34.2 ms <

 Memory estimate: 8.94 MiB, allocs estimate: 47993.

In [13]:
@time pdp(params, "vf");

iteration counter: 205
error (value): 9.667884398985365e-6
error (policy): 1.2621223282946012e-7
  0.044092 seconds (66.80 k allocations: 10.063 MiB, 13.98% compilation time)


In [15]:
@benchmark pdp(params, "pf"; verbosity=0)

BenchmarkTools.Trial: 1627 samples with 1 evaluation.
 Range (min … max):  2.989 ms …   5.191 ms  ┊ GC (min … max): 0.00% … 40.65%
 Time  (median):     3.042 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.072 ms ± 163.955 μs  ┊ GC (mean ± σ):  0.55% ±  3.41%

  ▁▇█▆▄▂▁▂                                                     
  ██████████▆█▆▇▃▆▄▄▁▃▄▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▃▃▁▃ █
  2.99 ms      Histogram: log(frequency) by time       4.2 ms <

 Memory estimate: 571.72 KiB, allocs estimate: 2871.

In [54]:
@time pdp(params, "pf");

iteration counter: 13
error (value): 0.0632195227136131
error (policy): 4.024750452184732e-6
  0.006343 seconds (2.96 k allocations: 575.359 KiB)


In [48]:
@time vfcn0, pfcn0, dif = pdp(params, "vf");

iteration counter: 205
error (value): 9.667884398985365e-6
error (policy): 1.2621223282946012e-7
  0.036606 seconds (48.06 k allocations: 8.947 MiB)


---

In [49]:
# 最終的な政策関数が得られてから消費関数を計算
wealth = params.kgrid.^params.α + (1-params.δ)*params.kgrid
cfcn = wealth - pfcn0;

In [50]:
# 政策関数を使って収束した価値関数を計算
util = zeros(params.nk)
valfn = zeros(params.nk)

for i = 1:params.nk
    cons = params.kgrid[i]^params.α + (1-params.δ)*params.kgrid[i] - pfcn0[i]
    util[i] = MyEconFcn.crra(cons, params.γ)
    valfn[i] = util[i] / (1-params.β)
end

In [51]:
# 解析解
AA = (1-params.β)^(-1) * (log(1-params.α*params.β) + ((params.α*params.β)/(1-params.α*params.β))*log(params.α*params.β))
BB = params.α / (1 - params.α*params.β)
v_true = AA .+ BB*log.(params.kgrid)
p_true = params.α*params.β*(params.kgrid.^params.α);

---

# プロット

In [52]:
plt = plot(params.kgrid, valfn,
    color = :blue,
    legend = :none,
    xlabel = ("現在の資本：k"),
    ylabel = ("価値関数：V(k)"),
    linewidth = 4,
    titlefont = font("HackGen35Nerd", 12),
    guidefont = font("HackGen35Nerd", 12),
    tickfont = font("HackGen35Nerd", 8),
    legend_font_family = ("HackGen35Nerd"),
    legendfontsize = 12,
    framestyle = :semi,
)

┌ Warning: Framestyle :semi is not supported by Plotly and PlotlyJS. :box was chosen instead.
└ @ Plots ~/.julia/packages/Plots/GIume/src/backends/plotly.jl:8


In [54]:
plt = plot(params.kgrid, pfcn0,
    color = :blue,
    legend = :topleft,
    label = ("近似解"),
    xlabel = ("現在の資本：k"),
    ylabel = ("次期の資本：k'=g(k)"),
    linewidth = 4,
    titlefont = font("HackGen35Nerd", 12),
    guidefont = font("HackGen35Nerd", 12),
    tickfont = font("HackGen35Nerd", 8),
    legend_font_family = ("HackGen35Nerd"),
    legendfontsize = 12,
    framestyle = :semi
)
plot!(params.kgrid, p_true, linewidth = 1, label="解析解")
plot!(params.kgrid, params.kgrid, linewidth = 3, color = "black", linestyle = :dot, label="45度線")

┌ Warning: Framestyle :semi is not supported by Plotly and PlotlyJS. :box was chosen instead.
└ @ Plots ~/.julia/packages/Plots/GIume/src/backends/plotly.jl:8


In [45]:
time = 1:205

plt = plot(time, dif[1, 1:205],
    color = :blue,
    legend = :none,
    xlabel = ("Iteration"),
    ylabel = ("Error"),
    linewidth = 4,
    titlefont = font("HackGen35Nerd", 12),
    guidefont = font("HackGen35Nerd", 12),
    tickfont = font("HackGen35Nerd", 8),
    legend_font_family = ("HackGen35Nerd"),
    legendfontsize = 12,
    framestyle = :semi
)

┌ Warning: Framestyle :semi is not supported by Plotly and PlotlyJS. :box was chosen instead.
└ @ Plots ~/.julia/packages/Plots/GIume/src/backends/plotly.jl:8


In [46]:
time = 1:205

plt = plot(time, dif[2, 1:205],
    color = :blue,
    legend = :none,
    xlabel = ("Iteration"),
    ylabel = ("Error"),
    linewidth = 4,
    titlefont = font("HackGen35Nerd", 12),
    guidefont = font("HackGen35Nerd", 12),
    tickfont = font("HackGen35Nerd", 8),
    legend_font_family = ("HackGen35Nerd"),
    legendfontsize = 12,
    framestyle = :semi
)

┌ Warning: Framestyle :semi is not supported by Plotly and PlotlyJS. :box was chosen instead.
└ @ Plots ~/.julia/packages/Plots/GIume/src/backends/plotly.jl:8


In [47]:

time = 1:205

plot(time, dif[1, 1:205],
    color = :blue,
    legend = :none,
    xlabel = ("Iteration"),
    ylabel = ("Error"),
    linewidth = 4,
    titlefont = font("HackGen35Nerd", 12),
    guidefont = font("HackGen35Nerd", 12),
    tickfont = font("HackGen35Nerd", 8),
    legend_font_family = ("HackGen35Nerd"),
    legendfontsize = 12,
    framestyle = :semi,
    ylim = [0, 0.1]
)

plot!(time, dif[2, 1:205],
    color = :gray,
    legend = :none,
    xlabel = ("Iteration"),
    ylabel = ("Error"),
    linewidth = 4,
    titlefont = font("HackGen35Nerd", 12),
    guidefont = font("HackGen35Nerd", 12),
    tickfont = font("HackGen35Nerd", 8),
    legend_font_family = ("HackGen35Nerd"),
    legendfontsize = 12,
    framestyle = :semi
)

┌ Warning: Framestyle :semi is not supported by Plotly and PlotlyJS. :box was chosen instead.
└ @ Plots ~/.julia/packages/Plots/GIume/src/backends/plotly.jl:8


---

## エクササイズ
* グリッドを等分から別の方法に切り替えてみよう。
    * Hint：GenerateGrid.jl内にある関数を使ってみる。
* 所得リスクを導入してみよう。